# Invoice extraction: LoRA fine-tuning on a free Colab GPU

Runtime > Change runtime type > **T4 GPU**. Runs the same scripts as the repo with a larger model (Qwen2.5-1.5B-Instruct). Roughly 15-25 minutes end to end.

In [ ]:
!git clone https://github.com/Sup3rMar10/llm-finetune-invoice-extraction.git
%cd llm-finetune-invoice-extraction
!pip -q install -r requirements.txt

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'  # use 0.5B for speed, 3B on an A100

In [ ]:
!python -m src.generate_data
!python -m src.train --model $MODEL --out outputs/lora-1.5b --epochs 2 --batch-size 8 --grad-accum 1 --grad-checkpointing

In [ ]:
runs = [('base-zeroshot', ''), ('base-fewshot', '--fewshot 2'), ('tuned', '--adapter outputs/lora-1.5b')]
for name, extra in runs:
    for split in ['test_id', 'test_ood']:
        !python -m src.evaluate --model $MODEL --name {name} --split data/{split}.jsonl {extra} --batch-size 16 --out results_colab

In [ ]:
!python -m src.report --results results_colab --log outputs/lora-1.5b/train_log.jsonl
print(open('results_colab/RESULTS.md').read())

In [ ]:
from google.colab import files
!zip -r lora-1.5b.zip outputs/lora-1.5b results_colab
files.download('lora-1.5b.zip')